# Fall Classification Training - Temporal RGB + EfficientNet-B0

**Goal:** Train a binary classifier (fall vs normal) using temporal RGB triplets

**Architecture:**
- Backbone: EfficientNet-B0 (pretrained on ImageNet)
- Input: 224x224 RGB images where:
  - Red channel = frame at t-1 (grayscale)
  - Green channel = frame at t (grayscale)
  - Blue channel = frame at t+1 (grayscale)
- Output: 2 classes (fall, normal)

**Training strategy:**
- Phase 1: Freeze backbone, train head only (5 epochs)
- Phase 2: Unfreeze all, fine-tune end-to-end (25 epochs)
- Early stopping on val F1 (patience=10)

**Expected results:** 90-95% accuracy, 88-95% fall recall

## Cell 1: Setup

In [ ]:
!pip install -q timm torch torchvision scikit-learn matplotlib seaborn

import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Paths (adjust if your dataset is in a different location on Kaggle)
DATA_ROOT = Path('/kaggle/input/fall-classification')
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_ROOT.mkdir(exist_ok=True)

## Cell 2: Load Dataset & Transforms

In [ ]:
# ImageNet normalization stats (EfficientNet was pretrained on ImageNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Data augmentation for training
# CRITICAL: hue/saturation MUST be 0 because color channels encode time, not color!
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.0,  # MUST be 0
        hue=0.0          # MUST be 0
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# No augmentation for val/test
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Load datasets
train_dataset = ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_dataset = ImageFolder(DATA_ROOT / 'val', transform=eval_transform)
test_dataset = ImageFolder(DATA_ROOT / 'test', transform=eval_transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")  # ['fall', 'normal']

# Class distribution
train_labels = [label for _, label in train_dataset.samples]
val_labels = [label for _, label in val_dataset.samples]
test_labels = [label for _, label in test_dataset.samples]

print(f"\nTrain class distribution:")
for class_name, class_idx in train_dataset.class_to_idx.items():
    count = train_labels.count(class_idx)
    print(f"  {class_name}: {count} ({100*count/len(train_labels):.1f}%)")

print(f"\nVal class distribution:")
for class_name, class_idx in val_dataset.class_to_idx.items():
    count = val_labels.count(class_idx)
    print(f"  {class_name}: {count} ({100*count/len(val_labels):.1f}%)")

print(f"\nTest class distribution:")
for class_name, class_idx in test_dataset.class_to_idx.items():
    count = test_labels.count(class_idx)
    print(f"  {class_name}: {count} ({100*count/len(test_labels):.1f}%)")

## Cell 3: WeightedRandomSampler (handles residual imbalance)

In [ ]:
# Compute class weights for weighted sampling (handles any residual imbalance)
class_counts = [train_labels.count(i) for i in range(len(train_dataset.classes))]
class_weights = [1.0 / count for count in class_counts]
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print(f"Class weights: {class_weights}")
print(f"Weighted sampling enabled for training")

## Cell 4: DataLoaders

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0  # Use 0 in Kaggle/Jupyter to avoid DataLoader worker shutdown errors

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## Cell 5: Model Definition

In [ ]:
# Create EfficientNet-B0 with pretrained ImageNet weights
model = timm.create_model(
    'efficientnet_b0',
    pretrained=True,
    num_classes=2  # fall, normal
)

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: EfficientNet-B0")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Cell 6: Training Configuration

In [ ]:
# Loss function with class weights (minority class = fall gets higher weight)
class_counts = [train_labels.count(i) for i in range(len(train_dataset.classes))]
n_total = sum(class_counts)
loss_weights = torch.tensor([n_total / (2 * c) for c in class_counts], dtype=torch.float32)
loss_weights = loss_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights)
print(f"Loss class weights (fall, normal): {loss_weights.tolist()}")

# Optimizer (will be reset for phase 1 and phase 2)
# Phase 1: head only (lr=1e-4)
# Phase 2: full model (lr=1e-5 for backbone, 1e-4 for head)

# Training config
PHASE1_EPOCHS = 5  # Freeze backbone
PHASE2_EPOCHS = 25  # Unfreeze all
EARLY_STOP_PATIENCE = 10
SAVE_PATH = OUTPUT_ROOT / 'best.pt'
LAST_PATH = OUTPUT_ROOT / 'last.pt'  # Periodic checkpoint (if Kaggle times out)

print(f"Phase 1 (frozen backbone): {PHASE1_EPOCHS} epochs")
print(f"Phase 2 (full fine-tuning): {PHASE2_EPOCHS} epochs")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE}")

## Cell 7: Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': 100.0 * correct / total
        })
    
    return running_loss / len(loader), 100.0 * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model and return metrics"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = running_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary', pos_label=0)  # 0=fall
    recall = recall_score(all_labels, all_preds, average='binary', pos_label=0)
    f1 = f1_score(all_labels, all_preds, average='binary', pos_label=0)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'preds': all_preds,
        'labels': all_labels,
        'probs': all_probs
    }

print("Training functions defined")

## Cell 8: Phase 1 - Train Head Only (Frozen Backbone)

In [ ]:
print("=" * 80)
print("PHASE 1: TRAINING HEAD ONLY (BACKBONE FROZEN)")
print("=" * 80)

# Freeze backbone (only train classifier head)
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}\n")

# Optimizer for phase 1
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_f1': []}

for epoch in range(PHASE1_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{PHASE1_EPOCHS}")
    
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['f1'])
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f}, Val F1: {val_metrics['f1']:.4f}, Val Acc: {val_metrics['accuracy']*100:.2f}%")
    
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"✓ Saved best model (F1: {best_val_f1:.4f})")
    torch.save(model.state_dict(), LAST_PATH)  # Checkpoint every epoch

print(f"\nPhase 1 complete. Best val F1: {best_val_f1:.4f}")

## Cell 9: Phase 2 - Fine-tune Full Model

In [ ]:
print("\n" + "=" * 80)
print("PHASE 2: FINE-TUNING FULL MODEL (BACKBONE UNFROZEN)")
print("=" * 80)

# Unfreeze all parameters
for param in model.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}\n")

# Load best checkpoint from phase 1
model.load_state_dict(torch.load(SAVE_PATH))
print("✓ Loaded best checkpoint from Phase 1\n")

# Optimizer with differential learning rates
backbone_params = []
head_params = []
for name, param in model.named_parameters():
    if 'classifier' in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},  # Lower LR for backbone
    {'params': head_params, 'lr': 1e-4}       # Higher LR for head
], weight_decay=0.01)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

best_val_f1 = 0.0
patience_counter = 0

for epoch in range(PHASE2_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{PHASE2_EPOCHS}")
    
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_f1'].append(val_metrics['f1'])
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f}, Val F1: {val_metrics['f1']:.4f}, Val Acc: {val_metrics['accuracy']*100:.2f}%")
    print(f"Val Precision: {val_metrics['precision']:.4f}, Val Recall (Fall): {val_metrics['recall']:.4f}")
    
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"✓ Saved best model (F1: {best_val_f1:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")
        
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch + 1}")
            break
    torch.save(model.state_dict(), LAST_PATH)  # Checkpoint every epoch

print(f"\nPhase 2 complete. Best val F1: {best_val_f1:.4f}")

## Cell 10: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].axvline(x=PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.5, label='Phase 2 starts')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1
axes[1].plot(history['val_f1'], label='Val F1', color='green')
axes[1].axvline(x=PHASE1_EPOCHS, color='red', linestyle='--', alpha=0.5, label='Phase 2 starts')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'training_history.png', dpi=150)
plt.show()

print("✓ Training history plot saved")

## Cell 11: Final Evaluation on Test Set

In [ ]:
print("=" * 80)
print("FINAL EVALUATION ON TEST SET")
print("=" * 80)

# Load best model
model.load_state_dict(torch.load(SAVE_PATH))
print("✓ Loaded best model\n")

# Evaluate on test set
test_metrics = evaluate(model, test_loader, criterion, device)

print(f"\nTest Set Results:")
print(f"  Accuracy: {test_metrics['accuracy']*100:.2f}%")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall (Fall): {test_metrics['recall']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(
    test_metrics['labels'],
    test_metrics['preds'],
    target_names=['fall', 'normal']
))

## Cell 12: Confusion Matrix

In [ ]:
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'])

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['fall', 'normal'],
    yticklabels=['fall', 'normal']
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'confusion_matrix.png', dpi=150)
plt.show()

print("✓ Confusion matrix saved")

## Cell 13: ROC Curve

In [ ]:
# ROC curve for fall detection (class 0)
probs_array = np.array(test_metrics['probs'])
fall_probs = probs_array[:, 0]  # Probability of fall
binary_labels = [1 if label == 0 else 0 for label in test_metrics['labels']]  # 1=fall, 0=normal

fpr, tpr, thresholds = roc_curve(binary_labels, fall_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Fall Recall)')
plt.title('ROC Curve - Fall Detection')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'roc_curve.png', dpi=150)
plt.show()

print(f"✓ ROC curve saved (AUC: {roc_auc:.3f})")

## Cell 14: Sample Predictions

In [ ]:
# Visualize some sample predictions
num_samples = 16
sample_indices = random.sample(range(len(test_dataset)), num_samples)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

model.eval()
with torch.no_grad():
    for idx, sample_idx in enumerate(sample_indices):
        image, true_label = test_dataset[sample_idx]
        image_tensor = image.unsqueeze(0).to(device)
        
        output = model(image_tensor)
        prob = torch.softmax(output, dim=1)
        pred_label = output.argmax(dim=1).item()
        
        # Denormalize for visualization
        img_show = image.permute(1, 2, 0).cpu().numpy()
        img_show = img_show * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_show = np.clip(img_show, 0, 1)
        
        axes[idx].imshow(img_show)
        axes[idx].axis('off')
        
        true_class = test_dataset.classes[true_label]
        pred_class = test_dataset.classes[pred_label]
        confidence = prob[0, pred_label].item()
        
        color = 'green' if pred_label == true_label else 'red'
        axes[idx].set_title(
            f"True: {true_class}\nPred: {pred_class} ({confidence:.2f})",
            fontsize=8,
            color=color
        )

plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'sample_predictions.png', dpi=150)
plt.show()

print("✓ Sample predictions saved")

## Cell 15: Save Metrics as JSON

In [ ]:
metrics_dict = {
    'test_accuracy': float(test_metrics['accuracy']),
    'test_precision': float(test_metrics['precision']),
    'test_recall_fall': float(test_metrics['recall']),
    'test_f1': float(test_metrics['f1']),
    'roc_auc': float(roc_auc),
    'confusion_matrix': cm.tolist(),
    'training_config': {
        'model': 'efficientnet_b0',
        'phase1_epochs': PHASE1_EPOCHS,
        'phase2_epochs': PHASE2_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr_head': 1e-4,
        'lr_backbone': 1e-5,
        'weight_decay': 0.01,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'seed': SEED
    },
    'dataset_info': {
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'test_samples': len(test_dataset),
        'classes': train_dataset.classes
    }
}

with open(OUTPUT_ROOT / 'metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print("✓ Metrics saved to metrics.json")
print("\n" + json.dumps(metrics_dict, indent=2))

## Cell 16: Export Model & Create Download Archive

In [ ]:
import shutil

# Create export directory
export_dir = OUTPUT_ROOT / 'fall_classifier_export'
export_dir.mkdir(exist_ok=True)

# Copy model weights
shutil.copy(SAVE_PATH, export_dir / 'best.pt')

# Copy metrics and plots
shutil.copy(OUTPUT_ROOT / 'metrics.json', export_dir / 'metrics.json')
shutil.copy(OUTPUT_ROOT / 'training_history.png', export_dir / 'training_history.png')
shutil.copy(OUTPUT_ROOT / 'confusion_matrix.png', export_dir / 'confusion_matrix.png')
shutil.copy(OUTPUT_ROOT / 'roc_curve.png', export_dir / 'roc_curve.png')
shutil.copy(OUTPUT_ROOT / 'sample_predictions.png', export_dir / 'sample_predictions.png')

# Save training history
with open(export_dir / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

# Create README
readme_content = f"""# Fall Classifier - EfficientNet-B0

Trained on temporal RGB triplets for fall detection.

## Results

- Test Accuracy: {test_metrics['accuracy']*100:.2f}%
- Test Precision: {test_metrics['precision']:.4f}
- Test Recall (Fall): {test_metrics['recall']:.4f}
- Test F1 Score: {test_metrics['f1']:.4f}
- ROC AUC: {roc_auc:.3f}

## Files

- `best.pt` - Model weights (EfficientNet-B0)
- `metrics.json` - Detailed metrics
- `training_history.json` - Loss/F1 per epoch
- `*.png` - Evaluation plots

## Usage

```python
import torch
import timm

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=2)
model.load_state_dict(torch.load('best.pt'))
model.eval()
```
"""

with open(export_dir / 'README.md', 'w') as f:
    f.write(readme_content)

# Create zip archive
shutil.make_archive(str(OUTPUT_ROOT / 'fall_classifier_export'), 'zip', export_dir)

print("=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)
print(f"\n✓ Model and artifacts exported to: {export_dir}")
print(f"✓ Archive created: fall_classifier_export.zip")
print(f"\nDownload fall_classifier_export.zip and extract best.pt to:")
print(f"  d:/project/FYP/fall_detection/weights/best.pt")
print(f"\nNext: Build visual_guardian module and integrate this model")